<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Install Libraries
</h2>

In [1]:
!pip install langchain-community faiss-cpu langchain-core groq tabula-py requests pillow transformers torch pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing 

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Importing the libs
</h2>

In [2]:
import faiss
import json
import base64
import pdfplumber
from langchain.text_splitter import RecursiveCharacterTextSplitter
import requests
import os
import logging
import numpy as np
import warnings
from tqdm import tqdm
from typing import List, Dict, Any

from PIL import Image
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer
from groq import Groq

logger = logging.getLogger(__name__)
logger.setLevel(logging.ERROR)

warnings.filterwarnings("ignore")

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Set Up Credentials & Models
</h2>

In [5]:
GROQ_API_KEY = "GROQ_API_KEY"

if GROQ_API_KEY is None:
    print("Warning: GROQ_API_KEY environment variable not set. Please set it to use the Groq LLM.")
else:
    groq_client = Groq(api_key=GROQ_API_KEY)

In [6]:
# Load HuggingFace CLIP model and processor
print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Loading Jina v4 embedding model for text/tables/images...")
#embedding_model_id = "jinaai/jina-embeddings-v4-vllm-retrieval"
#embed_model = SentenceTransformer(embedding_model_id, trust_remote_code=True)
embedding_model_id = "abhinand/MedEmbed-large-v0.1"
embed_model = SentenceTransformer(embedding_model_id)

Loading CLIP model...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Embedding Functions
</h2>

In [7]:
def get_text_embedding(text: str) -> np.ndarray:
    """Generates embeddings for the given text using Jina AI model."""
    try:
        emb = embed_model.encode([text], batch_size=64,
                                        convert_to_numpy=True, normalize_embeddings=True)[0]
        #Jina encode fucntion
        #emb = embed_model.encode([text], convert_to_numpy=True, show_progress_bar=False, task='retrieval')[0]
        return emb.astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating text/table embedding: {e}")
        return None

def get_image_embedding(image_path: str) -> np.ndarray:
    """Generates embeddings for an image using HuggingFace CLIP's image encoder."""
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors='pt')
        image_features = clip_model.get_image_features(**inputs)
        return image_features.detach().numpy().flatten().astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating image embedding for {image_path}: {e}")
        return None
def get_clip_text_embedding(text: str) -> np.ndarray:
    """Generates embeddings for text using HuggingFace CLIP's text encoder."""
    try:
        inputs = clip_processor(text=text, return_tensors='pt', padding=True, truncation=True)
        text_features = clip_model.get_text_features(**inputs)
        return text_features.detach().numpy().flatten().astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating CLIP text embedding: {e}")
        return None


In [8]:
#CLIP Text Embedder
def get_clip_text_embedding(text: str) -> np.ndarray:
    """Generates embeddings for text using HuggingFace CLIP's text encoder."""
    try:
        inputs = clip_processor(text=text, return_tensors='pt', padding=True, truncation=True)
        text_features = clip_model.get_text_features(**inputs)
        return text_features.detach().numpy().flatten().astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating CLIP text embedding: {e}")
        return None

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Data Loading
</h2>

In [9]:
import os

# Set the filename and filepath (already in data/)
filename = "Prot_000.pdf"
filepath = os.path.join(filename)

# Make sure the file exists
if not os.path.exists(filepath):
    raise FileNotFoundError(f"{filepath} not found! Please upload it to the data/ folder.")
else:
    print(f"Using file: {filepath}")


Using file: Prot_000.pdf


<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Data Extraction
</h2>

In [15]:
def create_directories(base_dir):
    directories = ["images", "text", "tables"]
    for d in directories:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def _pdf_bbox_to_pixels(bbox, page_width, page_height, dpi=300):
    """
    Convert PDF bbox (x0,y0,x1,y1) in points (origin bottom-left) to
    PIL pixel box (left, top, right, bottom) with origin top-left.
    """
    x0, y0, x1, y1 = bbox
    scale = dpi / 72.0
    left   = int(round(x0 * scale))
    right  = int(round(x1 * scale))
    top    = int(round((page_height - y1) * scale))
    bottom = int(round((page_height - y0) * scale))
    return (left, top, right, bottom)

def process_tables(pdf, page_num, base_dir, items, filepath):
    page = pdf.pages[page_num]

    # Try a conservative line-based config first
    settings_try = [
        # 1) lines-based detection (works for ruled tables)
        {"vertical_strategy": "lines", "horizontal_strategy": "lines"},
        # 2) text-based detection (works for unruled tables)
        {"vertical_strategy": "text", "horizontal_strategy": "text",
         "text_x_tolerance": 2, "text_y_tolerance": 2},
    ]

    tables = []
    for ts in settings_try:
        try:
            tables = page.extract_tables(table_settings=ts) or []
            if tables:
                break
        except TypeError as e:
            # If this version of pdfplumber doesn’t accept some keys, retry with minimal keys
            if ts.get("vertical_strategy") == "text":
                try:
                    tables = page.extract_tables(
                        table_settings={"vertical_strategy": "text", "horizontal_strategy": "text"}
                    ) or []
                    if tables:
                        break
                except Exception:
                    pass
        except Exception:
            # ignore and try the next strategy
            pass

    if not tables:
        return  # nothing found on this page

    for table_idx, table in enumerate(tables):
        # table is a list of rows (lists of cell strings or None)
        lines = []
        for row in table or []:
            row = row or []
            cells = ["" if c is None else str(c).strip() for c in row]
            lines.append(" | ".join(cells))
        table_text = "\n".join(lines)

        out_path = f"{base_dir}/tables/{os.path.basename(filepath)}_table_{page_num}_{table_idx}.txt"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(table_text)

        items.append({"page": page_num, "type": "table", "text": table_text, "path": out_path})

def process_text_chunks(text, page_num, base_dir, items, filepath):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=200)
    chunks = text_splitter.split_text(text or "")
    for i, chunk in enumerate(chunks):
        text_file_name = f"{base_dir}/text/{os.path.basename(filepath)}_text_{page_num}_{i}.txt"
        with open(text_file_name, 'w', encoding="utf-8") as f:
            f.write(chunk)
        items.append({"page": page_num, "type": "text", "text": chunk, "path": text_file_name})

def process_images(pdf, page_num, base_dir, items, filepath, dpi=300):
    page = pdf.pages[page_num]
    images = page.images or []
    if not images:
        return
    # Render page once at chosen DPI to crop from
    pil_page = page.to_image(resolution=dpi).original  # PIL.Image
    for idx, im in enumerate(images):
        bbox_pts = (im["x0"], im["y0"], im["x1"], im["y1"])
        crop_box = _pdf_bbox_to_pixels(bbox_pts, page.width, page.height, dpi=dpi)
        l, t, r, b = crop_box
        if r - l <= 1 or b - t <= 1:
            continue
        cropped = pil_page.crop(crop_box)
        if cropped.mode not in ("RGB", "RGBA"):
            cropped = cropped.convert("RGB")
        image_name = f"{base_dir}/images/{os.path.basename(filepath)}_image_{page_num}_{idx}.png"
        cropped.save(image_name, format="PNG")
        items.append({"page": page_num, "type": "image", "path": image_name})

def extract_items_from_pdf(filepath):
    base_dir = "data"
    create_directories(base_dir)
    items = []

    with pdfplumber.open(filepath) as pdf:
        num_pages = len(pdf.pages)
        for page_num in tqdm(range(num_pages), desc="Processing PDF pages"):
            page = pdf.pages[page_num]
            # Text
            text = page.extract_text() or ""
            process_text_chunks(text, page_num, base_dir, items, filepath)
            # Tables
            process_tables(pdf, page_num, base_dir, items, filepath)
            # Images (cropped from rendered page using image bboxes)
            process_images(pdf, page_num, base_dir, items, filepath)

    return items

In [16]:
items = extract_items_from_pdf(filepath)

Processing PDF pages: 100%|██████████| 233/233 [00:42<00:00,  5.51it/s]


In [17]:
import os

base_dir = "data"
subfolders = ["images", "tables", "text"]

for sub in subfolders:
    folder_path = os.path.join(base_dir, sub)
    if os.path.exists(folder_path):
        files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
        print(f"{sub}: {len(files)} files")
    else:
        print(f"{sub}: folder not found")


images: 19 files
tables: 242 files
text: 1037 files


In [18]:
# Looking at the first text item
[i for i in items if i['type'] == 'text'][0]

{'page': 0,
 'type': 'text',
 'text': 'Official Title of Study:\nA Phase 1/2 Study of the Combination of Lirilumab (Anti-KIR) Plus Nivolumab (Anti-PD-1) or Lirilumab\nPlus Nivolumab and Ipilimumab in Advanced Refractory Solid Tumors\nNCT Number: NCT01714739\nDocument Date (Date in which document was last revised): May 8, 2018',
 'path': 'data/text/Prot_000.pdf_text_0_0.txt'}

In [19]:
# Looking at the first table item
[i for i in items if i['type'] == 'table'][0]

{'page': 0,
 'type': 'table',
 'text': 'e of Study:\n\n(Anti-KIR) Plu\n\nAdvanced Ref\n\nNCT01714739\n\nment was last',
 'path': 'data/tables/Prot_000.pdf_table_0_0.txt'}

In [20]:
# Looking at the first image item
[i for i in items if i['type'] == 'image'][0]

{'page': 18,
 'type': 'image',
 'path': 'data/images/Prot_000.pdf_image_18_0.png'}

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Vector Search and Retrieval
</h2>

In [21]:
# Generate embeddings and prepare documents for vector stores
text_embeddings = []
text_docs = []
image_embeddings = []
image_docs = []

print("Generating embeddings for text and table items...")
for item in tqdm(items, desc="Processing items"):
    if item['type'] in ['text', 'table']:
        embedding = get_text_embedding(item['text'])
        if embedding is not None:
            text_embeddings.append(embedding)
            text_docs.append(item)
    elif item['type'] == 'image':
        embedding = get_image_embedding(item['path'])
        if embedding is not None:
            image_embeddings.append(embedding)
            image_docs.append(item)

# Create FAISS indices
print("Creating text and image FAISS indices...")
text_embedding_dim = text_embeddings[0].shape[0]
text_index = faiss.IndexFlatL2(text_embedding_dim)
text_index.add(np.array(text_embeddings))

image_embedding_dim = image_embeddings[0].shape[0]
image_index = faiss.IndexFlatL2(image_embedding_dim)
image_index.add(np.array(image_embeddings))

Generating embeddings for text and table items...


Processing items: 100%|██████████| 1298/1298 [00:37<00:00, 34.94it/s]

Creating text and image FAISS indices...


<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   RAG Pipeline with Groq
</h2>

In [28]:
def invoke_groq_multimodal(query: str, retrieved_items: List[Dict[str, Any]]) -> str:
    """Generates a response from the retrieved context using Groq LLM."""
    context = ""

    for item in retrieved_items:
        if item['type'] in ['text', 'table']:
            context += f"\n\nText from page {item['page']}:\n{item['text']}"
        elif item['type'] == 'image':
            context += f"\n\nImage from page {item['page']}:\nPath: {item['path']}"

    prompt_template = f"""You are a helpful assistant. Use the following pieces of text and image information to answer the user's question. If the answer is not in the provided context, politely say that you cannot provide an answer. Do not use any external knowledge. If an image is relevant, mention its file path. \n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""

    messages = [
        {"role": "user", "content": prompt_template}
    ]

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=2048,
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error invoking Groq: {str(e)}"

def rag_pipeline(query: str):
    # Get embeddings for the query
    text_query_embedding = get_text_embedding(query)

    # The image search must use the CLIP text encoder to embed the query
    image_query_embedding = get_clip_text_embedding(query)

    retrieved_items = []

    # A specific check for the user's direct command to list all tables
    if query.lower().strip() == "show all the tables in this file":
        retrieved_items = [item for item in items if item['type'] == 'table']

    # If not a specific command, proceed with the normal RAG pipeline
    else:
        if text_query_embedding is not None:
            # Search text index
            text_distances, text_results = text_index.search(np.array([text_query_embedding]), k=3)
            for idx in text_results.flatten():
                retrieved_items.append(text_docs[idx])

        if image_query_embedding is not None:
            # Search image index
            image_distances, image_results = image_index.search(np.array([image_query_embedding]), k=2)
            for idx in image_results.flatten():
                retrieved_items.append(image_docs[idx])

    # Pass combined context to LLM
    if retrieved_items:
        response = invoke_groq_multimodal(query, retrieved_items)
        return response
    else:
        return "No relevant items found."

print(rag_pipeline("show a table from the pdf"))

There are several tables mentioned in the provided context. Here are a few examples:


Table 5.1.1 from page 120:


| Procedure | Visit | Notes |
| --- | --- | --- |
| Screening | D -28 to -1 | Collected for the period 2 weeks prior to Cycle 1 Day 1. |
| Assessment of Signs and Symptoms | X | Collected for the period 2 weeks prior to Cycle 1 Day 1. |
| Adverse Event Reporting |  | All SAEs must be collected starting at the time the subject signs informed consent and through 150 days after discontinuation of dosing. |
| ... | ... | ... |


Table 5.1.1-2 from page 124:


| Procedure | D1 | D15 | D29 | D43 | D49 to 56 | Notes |
| --- | --- | --- | --- | --- | --- | --- |
| Sample Collection |  |  |  |  |  | See Table 5.7.3.3-1, Table 5.7.3.3-2, Table 5.7.3.3-3, and Table 5.7.3.3-4. |
| PK Assessments |  |  |  |  |  | To be collected on Days 1, 2, 8, 15, 29, 85, 86, 92, and 113. |
| ... | ... | ... | ... | ... | ... | ... |


Table 1 from page 226:


| Target lesions | Non-target lesions |

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Thank you!
</h2>

In [ ]:
items = extract_items_from_pdf(filepath)

Processing PDF pages:   0%|          | 1/233 [00:01<03:54,  1.01s/it]WARNING:tabula.backend:Got stderr: Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile

Processing PDF pages:   1%|          | 2/233 [00:02<04:46,  1.24s/it]WARNING:tabula.backend:Got stderr: Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile

Processing PDF

images: 19 files
tables: 0 files
text: 1037 files
